# Large-scale OR: Few-shot-only

Two-stage GPT-4.1 workflow without CSVQA or FAISS. The classifier receives five fixed question/category records copied from RefData.csv. The modeler receives one worked case for NRM, RA, TP, FLP, or AP, and three for Others. Worked-case CSVs and current-case CSVs are attached to the API request as complete text files.


In [ ]:
from __future__ import annotations

import csv, hashlib, json, math, os, re, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path(os.environ.get('LEAN_LLM_OPT_ROOT', Path.cwd())).resolve()
BENCHMARK = ROOT / 'Test_Dataset/Large-scale-or/Large-scale-or-101.csv'
MODEL = 'gpt-4.1-2025-04-14'
if not BENCHMARK.is_file():
    raise FileNotFoundError('Run from the repository root or set LEAN_LLM_OPT_ROOT')

CLASS_RULES = '''Return exactly one label: NRM, RA, TP, FLP, AP, Mixture, or Others.
NRM allocates finite inventory or capacity to demand for revenue.
RA allocates resources among activities in a primarily single-stage model.
TP is source-to-destination shipment with supply, demand, and route costs.
FLP links facility-opening decisions to assignment or shipment.
AP is one-to-one matching or assignment.
Mixture has materially coupled modeling mechanisms or multiple stages.
Others is a coherent optimization problem outside those classes.
Use the current question's objective, variables, domains, and constraints.
Output the label token alone.'''

OUTPUT_RULES = '''Return exactly two sections:
### Mathematical Model
Specify sets, parameters, variables and their domains, objective, and all constraints.
Use an LP or MILP as required by the current question.
### Python Code
Provide exactly one fenced Python code block. It must be complete gurobipy
code, create a model named m or model, call optimize(), and leave that solved
model available at top level. Read only the current case CSV paths listed in
this request. Derive identifiers, dimensions, and coefficients from these files.
The code must solve from the repository root; do not invent file paths.'''

def resolve_csv_paths(address):
    out=[]
    for raw in str(address).splitlines():
        raw=raw.strip()
        if not raw:continue
        path=Path(raw)
        path=(path if path.is_absolute() else ROOT/path).resolve()
        try:path.relative_to(ROOT)
        except ValueError as exc:raise ValueError(f'Path leaves repository: {raw}') from exc
        if path.suffix.lower()!='.csv' or not path.is_file():
            raise FileNotFoundError(path)
        out.append(path)
    if not out:raise ValueError('No CSV files listed for this case')
    return out

def parse_objective(raw):
    text=str(raw).strip().replace(',','').replace('$','')
    m=re.search(r'Optimal\s+objective\s*[:=]?\s*([-+\d.eE]+)',text,re.I)
    value=float(m.group(1) if m else text)
    if not math.isfinite(value):raise ValueError(raw)
    return value

def gold_class(raw):
    text=str(raw).strip().upper()
    if text=='UFLP':return 'FLP'
    if text.startswith('OTHERS'):return 'Others'
    if text=='MIXTURE':return 'Mixture'
    if text in {'NRM','RA','TP','FLP','AP'}:return text
    raise ValueError(raw)

def route(label):
    if label in {'Mixture','Others'}:return 'Others'
    if label not in {'NRM','RA','TP','FLP','AP'}:raise ValueError(f'Bad class: {label!r}')
    return label

def cases():
    with BENCHMARK.open(encoding='utf-8-sig',newline='') as f:
        rows=list(csv.DictReader(f))
    if len(rows)!=101:raise ValueError(f'Expected 101 cases, found {len(rows)}')
    result=[]
    for i,row in enumerate(rows):
        result.append({'row':i,'query':row['Query'],'type':row['Problem Type'],
                       'files':resolve_csv_paths(row['Dataset_address']),
                       'gold':parse_objective(row['Label-objective'])})
    return result

def rel(path):return str(path.relative_to(ROOT))

def check_files():
    data=cases()
    counts={n:sum(len(c['files'])==n for c in data) for n in (1,2,3)}
    print('Rows:',len(data),'Files per row:',counts)
    return data

def output_text(response):
    if getattr(response,'status',None)=='incomplete':
        raise RuntimeError(f'Incomplete model response: {response.incomplete_details}')
    text=str(getattr(response,'output_text','') or '').strip()
    if not text:raise RuntimeError('Empty model response')
    return text

def usage(response):
    u=getattr(response,'usage',None)
    return {'input':int(getattr(u,'input_tokens',0) or 0),
            'output':int(getattr(u,'output_tokens',0) or 0)}

def plain_call(client,system,user):
    response=client.responses.create(model=MODEL,temperature=0,max_output_tokens=32768,
        input=[{'role':'system','content':system},{'role':'user','content':user}])
    return output_text(response),[usage(response)]

def extract_code(answer):
    if '### Mathematical Model' not in answer or '### Python Code' not in answer:
        raise ValueError('Model/code sections missing')
    blocks=re.findall(r'```(?:python|py)\s*\n(.*?)\n```',answer,re.I|re.S)
    if len(blocks)!=1:raise ValueError(f'Expected one Python block, found {len(blocks)}')
    return blocks[0].strip()

SOLVER_FOOTER = '''
import gurobipy as _ab_gp
_ab_model=globals().get('m',globals().get('model'))
if not isinstance(_ab_model,_ab_gp.Model):
    raise RuntimeError('Code did not leave a Gurobi model in m or model')
if _ab_model.Status!=_ab_gp.GRB.OPTIMAL:
    raise RuntimeError(f'Gurobi status: {_ab_model.Status}')
print('ABLATION_OBJECTIVE='+repr(float(_ab_model.ObjVal)))
'''

def solve_code(source,timeout=300):
    run=subprocess.run([sys.executable,'-c',source+'\n'+SOLVER_FOOTER],
        cwd=ROOT,text=True,capture_output=True,timeout=timeout,check=False)
    transcript=(run.stdout or '')+'\n'+(run.stderr or '')
    if run.returncode:raise RuntimeError(transcript[-3000:])
    match=re.search(r'ABLATION_OBJECTIVE=([-+\d.eE]+)',transcript)
    if not match:raise RuntimeError('Objective missing: '+transcript[-3000:])
    return float(match.group(1)),transcript[-3000:]

def save_scores(folder,selected):
    latest={}
    for line in (folder/'records.jsonl').read_text(encoding='utf-8').splitlines():
        if line.strip():
            record=json.loads(line);latest[record['row']]=record
    rows=[latest[i] for i in sorted(latest)]
    fields=['run_id','method','dataset','instance_id','gold_type','pred_type',
            'correct_optimal','gold_objective','predicted_objective','status',
            'dataset_address','input_tokens','output_tokens','tool_calls','wall_s']
    with (folder/'scored.csv').open('w',encoding='utf-8',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader()
        for r in rows:
            writer.writerow({'run_id':folder.name,'method':METHOD,
                'dataset':'Large-Scale-OR','instance_id':r['row'],
                'gold_type':r['gold_type'],'pred_type':r.get('pred_type'),
                'correct_optimal':int(bool(r.get('correct_optimal'))),
                'gold_objective':r['gold_objective'],
                'predicted_objective':r.get('predicted_objective'),
                'status':r['status'],'dataset_address':'\n'.join(r['files']),
                'input_tokens':(r.get('usage') or {}).get('input',0),
                'output_tokens':(r.get('usage') or {}).get('output',0),
                'tool_calls':r.get('tool_calls',0),'wall_s':r['wall_s']})
    summary={'run_id':folder.name,'method':METHOD,'n':len(rows),
        'complete':set(latest)==set(selected),
        'correct_optimal':sum(bool(r.get('correct_optimal')) for r in rows),
        'classification_correct':sum(bool(r.get('classification_correct')) for r in rows),
        'failed':sum(r['status']!='ok' for r in rows),
        'input_tokens':sum((r.get('usage') or {}).get('input',0) for r in rows),
        'output_tokens':sum((r.get('usage') or {}).get('output',0) for r in rows)}
    (folder/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')


## Fixed worked cases

The Others arm uses the first three rows of its full-workflow example bank. RA and TP request k=3 in the full workflow, but their banks contain only one row each; their actual full-workflow example count is one.


In [ ]:
DEMONSTRATIONS = {'NRM': [{'question': 'The data of the store offers several styles of Nike shoes is provided in "Nike Shoes Sales.csv". Through the dataset, the revenue of each shoes is listed in the column "revenue". The demand for each style is independent. The store’s objective is to maximize the total expected revenue based on the fixed initial inventories of the Nike x Olivia Kim brand, which are detailed in column “inventory.” During the sales horizon, no replenishment is allowed, and there are no in-transit inventories. Customer arrivals, corresponding to demand for different styles of Nike shoes, occur in each period according to a Poisson process, with arrival rates specified in column “demand.” Moreover, the trade will be started only when the total demand is no less than 100 to ensure the trading efficiency. The decision variables y_i represent the number of customer requests the store intends to fulfill for Nike shoe style i, with each y_i being a positive integer.', 'model': '##### Objective Function: \n\n$\\quad \\quad \\max \\quad \\sum_i A_i \\cdot x_i$\n\n##### Constraints\n\n###### 1. Inventory Constraints: \n\n$\\quad \\quad x_i \\leq I_i, \\quad \\forall i $\n\n###### 2. Demand Constraints: \n\n$x_i \\leq d_i, \\quad \\forall i $\n\n###### 3. Variable Constraints: \n\n$x_i \\in \\mathbb Z, x_i \\geq 0, \\quad \\forall i $\n\n###### Retrieved Information\n$\\small I = [97, 240, 322, 281]$\n$\\small A =  [11197, 9097, 11197, 9995]$\n$\\small d =  [17, 26, 50, 53]$\n$\\small s = 100$', 'code': '', 'files': ['Large_Scale_Or_Files/NRM_example/nike_Shoes_Sales.csv']}], 'RA': [{'question': 'A supermarket needs to allocate various products, including high-demand items like the Sony Alpha Refrigerator, Sony Bravia XR, and Sony PlayStation 5, across different retail shelves. The product values and space requirements are provided in the "Products.csv" dataset. Additionally, the store has multiple shelves, each with a total space limit and specific space constraints for Sony and Apple products, as outlined in the "Capacity.csv" file. The goal is to determine the optimal number of units of each Sony product to place on each shelf to maximize total value while ensuring that the space used by Sony products on each shelf does not exceed the brand-specific limits. The decision variables x_ij represent the number of units of product i to be placed on shelf j.', 'model': '##### Objective Function: \n\n$\\quad \\quad \\max \\sum_i \\sum_j p_i \\cdot x_{ij}$\n\n##### Constraints\n\n###### 1. Capacity Constraints: \n\n$\\quad \\quad \\sum_i a_i x_{ij} \\leq c_j, \\quad \\forall j$\n\n###### 2. Non-negativity Constraints: \n\n$\\quad \\quad  x_{ij} \\geq 0, \\quad \\forall i,j $\n\n###### Retrieved Information\np = [1818, 1609, 509, 1808, 528]\n\na = [400, 200, 40, 60, 50]\nc = [1200, 1374, 800, 2042, 1000, 1800]', 'code': '', 'files': ['Large_Scale_Or_Files/RA_example/capacity.csv', 'Large_Scale_Or_Files/RA_example/products.csv']}], 'TP': [{'question': 'Walmart stores play a critical role in supplying essential goods to various customer groups on a daily basis. The problem involves determining the optimal transportation plan that satisfies the demands of 4 customer groups, whose daily demand data is provided in "customer_demands.csv". These demands must be met using supplies from 3 Walmart stores, each with a specific daily supply capacity detailed in "supply_capacity.csv". The transportation cost per unit of goods from each Walmart store to each customer group is recorded in "transportation_costs.csv". The objective is to determine the quantity of goods transported from each Walmart store to each customer group, ensuring that all demands are met without exceeding the supply capacity of any store, while minimizing the total transportation cost.', 'model': '##### Objective Function: \n\n$\\quad \\quad \\min \\sum_i \\sum_j c_{ij} \\cdot x_{ij}$\n\n##### Constraints\n\n###### 1. Demand Constraints: \n\n$\\quad \\quad \\sum_i x_{ij} \\geq d_j, \\quad \\forall j$\n\n###### 2. Capacity Constraints: \n\n$\\quad \\quad  \\sum_j x_{ij} \\leq s_i, \\quad \\forall i$\n\n###### Retrieved Information\n```python\n$d = [11, 1148, 54, 833]$\n$s = [4, 575, 1504]$\n$ c = \\\\begin{bmatrix}\n0.63 & 49.71 & 33.75 & 1570.67 \\\\\n605.47 & 64.53 & 478.47 & 887.04 \\\\\n1139.04 & 4.78 & 1805.62 & 1302.89\n\\\\end{bmatrix}$\n```', 'code': '', 'files': ['Large_Scale_Or_Files/TP_example/customer_demand.csv', 'Large_Scale_Or_Files/TP_example/supply_capacity.csv', 'Large_Scale_Or_Files/TP_example/transportation_costs.csv']}], 'FLP': [{'question': "In the Superstore chain, multiple branches require inventory replenishment, and several suppliers located in different cities can provide the necessary goods. Each supplier incurs a fixed cost upon starting operations, with the fixed cost data provided in the 'fixed_costs.csv' file. Each branch needs to source a certain quantity of goods from these suppliers. For each branch, the transportation cost per unit of goods from each supplier is recorded in the 'transportation_costs.csv' file. Demand information can be gained in 'demand.csv'. The objective is to determine which suppliers to activate so that the demand of all branches is met while minimizing the total cost.", 'model': '##### Objective Function: \n\n$\\quad \\quad \\min \\sum_{i} \\sum_{j} A_{ij} x_{ij} + \\sum_{j} c_j y_j $\n\n##### Constraints\n\n###### 1. Demand Constraints: \n\n$\\quad \\quad \\sum_i x_{ij} = d_j, \\quad \\forall j$\n\n###### 2. M Constraints: \n\n$\\quad \\quad \\sum_i x_{ij} \\leq M y_j, \\quad \\forall j$\n\n###### 3. Variable Constraints: \n\n$ y_j \\in \\{{0,1\\\\}, \\quad \\forall j$\n\n$ x_{ij} \\geq 0, \\forall i,j $\n###### Retrieved Information\n\n$d = [7564, 5246]$\n$c = [105.1815083, 112.18423885]$\n$M = \\sum_j d_j = 7564 + 5246 = 12810$\n$ A = \\\\begin{bmatrix}\n937.34065048 & 86.93019411 \\\\\n49.38016148 & 1726.06210136\n\\\\end{bmatrix}$', 'code': '', 'files': ['Large_Scale_Or_Files/UFLP_example/demand.csv', 'Large_Scale_Or_Files/UFLP_example/fixed_cost.csv', 'Large_Scale_Or_Files/UFLP_example/transportation_costs.csv']}], 'AP': [{'question': "In the construction sector, there is a group of managers and a set of construction projects. The cost for a manager to oversee a specific project varies depending on their expertise and experience. This cost data is stored in a CSV file named 'manager_project_cost.csv', where each row corresponds to a manager and each column indicates the cost for that manager to handle a particular project. The aim is to determine the optimal assignment strategy that minimizes the overall cost of completing all projects. Each manager should be assigned to the project where their cost is lowest, ensuring that every project is managed by exactly one manager, and each manager is responsible for exactly one project. The goal is to minimize the total cost while adhering to the constraints that each project is assigned to one manager and each manager is assigned to one project", 'model': '##### Objective Function: \n\n$\\quad \\quad \\min \\sum_{i=1}^3 \\sum_{j=1}^3 c_{ij} x_{ij} $\n\n##### Constraints\n\n###### 1. Assignment Constraints: \n\n$\\sum_{j=1}^3 x_{ij} = 1 \\quad \\forall i \\in \\{1,2,3\\} $\n\n$\\sum_{i=1}^3 x_{ij} = 1 \\quad \\forall j \\in \\{1,2,3\\}$\n\n###### 2. Variable Constraints: \n\n$\\quad \\quad x_{ij} \\in \\{{0,1\\\\} \\quad \\forall i,j $\n\n###### Retrieved Information\n\n$c = \\\\begin{bmatrix}\n3000 & 3200 & 3100 \\\\\n2800 & 3300 & 2900 \\\\\n2900 & 3100 & 3000 \n\\\\end{bmatrix}  $', 'code': '', 'files': ['Large_Scale_Or_Files/AP_example/AP1/manager_project_cost.csv']}], 'Others': [{'question': 'A company plans to produce three products A_{1}, A_{2}, A_{3}. It can produce for 22 days in a month. The 36-1.csv file gives the maximum demand (in units of 100 kg), prices ($/100 kg), production costs (per 100 kg of product), and production quotas (the maximum number of units of 100 kg of product that can be produced in a day if all production lines are dedicated to that product).\n\nThe fixed activation cost for each production line is shown in 36-2.csv.\n\nThe minimum production batch sizes are shown in 36-3.csv.\n\nThe objective is to determine the production plan that maximizes total profit, accounting for total revenue, production costs, fixed activation costs, and minimum batch size restrictions.\n\nFurthermore, we only consider integer decision variables, meaning the quantities of products produced are integer multiples of 100 kg. The binary decisions on whether to activate each production line are also integer (0 or 1).', 'model': '## Mathematical Model\n\n**Maximize**\n\n$$\n50.7\\, x_1 - 170{,}000\\, y_1 \n+ 56.1\\, x_2 - 150{,}000\\, y_2 \n+ 49.6\\, x_3 - 100{,}000\\, y_3\n$$\n\n**subject to**\n\n$$\nx_1 \\leq 5300 \n\\quad (\\text{product 1 demand})\n$$\n\n$$\nx_2 \\leq 4500 \n\\quad (\\text{product 2 demand})\n$$\n\n$$\nx_3 \\leq 5400 \n\\quad (\\text{product 3 demand})\n$$\n\n$$\nx_1 \\leq 11{,}000 \n\\quad (\\text{product 1 monthly quota})\n$$\n\n$$\nx_2 \\leq 9{,}900 \n\\quad (\\text{product 2 monthly quota})\n$$\n\n$$\nx_3 \\leq 12{,}100 \n\\quad (\\text{product 3 monthly quota})\n$$\n\n$$\nx_1 \\geq 20\\, y_1 \n\\quad (\\text{minimum batch for product 1})\n$$\n\n$$\nx_2 \\geq 20\\, y_2 \n\\quad (\\text{minimum batch for product 2})\n$$\n\n$$\nx_3 \\geq 16\\, y_3 \n\\quad (\\text{minimum batch for product 3})\n$$\n\n$$\nx_1 \\leq 5300\\, y_1 \n\\quad (\\text{linking product 1 activation})\n$$\n\n$$\nx_2 \\leq 4500\\, y_2 \n\\quad (\\text{linking product 2 activation})\n$$\n\n$$\nx_3 \\leq 5400\\, y_3 \n\\quad (\\text{linking product 3 activation})\n$$\n\n**where**\n\n$$\nx_1, x_2, x_3 \\geq 0 \n\\quad (\\text{integer, production amount in 100kg units})\n$$\n\n$$\ny_1, y_2, y_3 \\in \\{0,1\\} \n\\quad (\\text{binary, activation of production line})\n$$', 'code': 'import gurobipy as gp\nimport pandas as pd\nimport numpy as np\nimport sys\nimport re\n\n# (Helper function from template - not used in this static model)\ndef find_col(df, pattern):\n    for col in df.columns:\n        if re.search(pattern, col, re.IGNORECASE):\n            return col\n    raise KeyError(f"Could not find a column matching pattern \'{{pattern}}\'")\n\ntry:\n    # 1. Load data\n    # (This step is skipped as all parameters are hard-coded\n    # in the mathematical model provided.)\n\n    # 2. Prepare data structures\n    # (Based on the static values from the problem description)\n    \n    products = [1, 2, 3]\n    \n    # Objective function coefficients\n    profit = {1: 50.7, 2: 56.1, 3: 49.6}\n    fixed_cost = {1: 170000, 2: 150000, 3: 100000}\n    \n    # Constraint parameters\n    demand = {1: 5300, 2: 4500, 3: 5400}\n    quota = {1: 11000, 2: 9900, 3: 12100}\n    min_batch = {1: 20, 2: 20, 3: 16}\n\n    # 3. Build Gurobi model\n    m = gp.Model(\'ProductionOptimization\')\n    \n    # 4. Define variables\n    # x[i]: production amount (in 100kg units) for product i\n    x = m.addVars(products, vtype=gp.GRB.INTEGER, name="x")\n    \n    # y[i]: binary activation variable for product i\n    y = m.addVars(products, vtype=gp.GRB.BINARY, name="y")\n    \n    # 5. Set objective\n    # Maximize 50.7*x1 - 170000*y1 + 56.1*x2 - 150000*y2 + ...\n    objective = gp.quicksum(profit[i] * x[i] - fixed_cost[i] * y[i] for i in products)\n    m.setObjective(objective, gp.GRB.MAXIMIZE)\n\n    # 6. Add constraints\n    \n    # Product demand constraints\n    # x1 <= 5300, x2 <= 4500, ...\n    m.addConstrs((x[i] <= demand[i] for i in products), name="demand")\n    \n    # Product monthly quota constraints\n    # x1 <= 11000, x2 <= 9900, ...\n    m.addConstrs((x[i] <= quota[i] for i in products), name="quota")\n    \n    # Minimum batch constraints\n    # x1 >= 20*y1, x2 >= 20*y2, ...\n    m.addConstrs((x[i] >= min_batch[i] * y[i] for i in products), name="min_batch")\n    \n    # Linking constraints (activation)\n    # x1 <= 5300*y1, x2 <= 4500*y2, ...\n    # (Note: These values match the \'demand\' dictionary)\n    m.addConstrs((x[i] <= demand[i] * y[i] for i in products), name="linking")\n    \n    # 7. Solve\n    m.optimize()\n    \n    # 8. Print results\n    if m.status == gp.GRB.OPTIMAL:\n        print(f"Optimal total value/cost: {m.objVal:.2f}")\n        print("--- Production Plan ---")\n        for i in products:\n            if y[i].X > 0.5: # Check if line is active\n                print(f"Product {i}:")\n                print(f"  Status: ACTIVE (y{i} = {y[i].X:.0f})")\n                print(f"  Production: {x[i].X:.0f} (100kg units)")\n            else:\n                print(f"Product {i}:")\n                print(f"  Status: INACTIVE (y{i} = {y[i].X:.0f})")\n                print(f"  Production: {x[i].X:.0f} (100kg units)")\n                \n    else:\n        print(f"No optimal solution found. Status: {m.status}")\n\nexcept KeyError as e:\n    print(f"Data Error: A required column was not found. {e}", file=sys.stderr)\nexcept pd.errors.EmptyDataError as e:\n    print(f"Data Error: CSV file is empty or unreadable. {e}", file=sys.stderr)\nexcept Exception as e:\n    print(f"An unexpected error occurred: {e}", file=sys.stderr)', 'files': ['Large_Scale_Or_Files/Other_example/36-1.csv', 'Large_Scale_Or_Files/Other_example/36-2.csv', 'Large_Scale_Or_Files/Other_example/36-3.csv']}, {'question': 'A factory plans to produce two products (A and B) using three limited resources: steel, iron, and machine hours. The resource consumption per unit of product and profit margins are shown in production_data.csv. Determine the optimal production quantities to maximize total profit while respecting resource constraints.', 'model': '#### Decision Variables\n- $x_1$: Quantity of Product A to produce\n- $x_2$: Quantity of Product B to produce\n\n#### Objective Function\nMaximize total profit: \n$\\quad \\quad \\max \\ 80x_1 + 125x_2$\n\n#### Constraints\n**1. Steel Constraint** \n$\\quad \\quad 8x_1 + 5x_2 \\leq 3500$ \n\n**2. Iron Constraint** \n$\\quad \\quad 6x_1 + 4x_2 \\leq 1800$ \n\n**3. Machine Hours Constraint** \n$\\quad \\quad 4x_1 + 5x_2 \\leq 2800$ \n\n**4. Non-negativity** \n$\\quad \\quad x_1 \\geq 0, x_2 \\geq 0$', 'code': 'import gurobipy as gp\nimport pandas as pd\nimport numpy as np\nimport sys\nimport re\n\n# Helper function to find columns robustly\n# (Not used in this static model, but included as part of the template)\ndef find_col(df, pattern):\n    for col in df.columns:\n        if re.search(pattern, col, re.IGNORECASE):\n            return col\n    raise KeyError(f"Could not find a column matching pattern \'{{pattern}}\'")\n\ntry:\n    # 1. Load data\n    # (This step is skipped as all parameters are hard-coded\n    # in the mathematical model provided.)\n    data_path = \'{dataset_address}\' # Not used\n\n    # 2. Prepare data structures\n    # (Skipped as all coefficients are used directly in the model)\n\n    # 3. Build Gurobi model\n    m = gp.Model(\'ProductionProfit\')\n    \n    # 4. Define variables\n    # x1: Quantity of Product A to produce\n    # x2: Quantity of Product B to produce\n    # Variables are continuous and non-negative by default (lb=0.0)\n    x1 = m.addVar(name="x1")\n    x2 = m.addVar(name="x2")\n    \n    # 5. Set objective\n    # Maximize total profit: 80*x1 + 125*x2\n    m.setObjective(80 * x1 + 125 * x2, gp.GRB.MAXIMIZE)\n\n    # 6. Add constraints\n    \n    # 1. Steel Constraint\n    # 8*x1 + 5*x2 <= 3500\n    m.addConstr(8 * x1 + 5 * x2 <= 3500, name="Steel")\n    \n    # 2. Iron Constraint\n    # 6*x1 + 4*x2 <= 1800\n    m.addConstr(6 * x1 + 4 * x2 <= 1800, name="Iron")\n    \n    # 3. Machine Hours Constraint\n    # 4*x1 + 5*x2 <= 2800\n    m.addConstr(4 * x1 + 5 * x2 <= 2800, name="MachineHours")\n    \n    # 4. Non-negativity (x1 >= 0, x2 >= 0)\n    # This is handled by default as the default lower bound (lb) for addVar is 0.0\n    \n    # 7. Solve\n    m.optimize()\n    \n    # 8. Print results\n    if m.status == gp.GRB.OPTIMAL:\n        print(f"Optimal total value/cost: {m.objVal:.2f}")\n        print("--- Optimal Production Plan ---")\n        print(f"Product A (x1): {x1.X:.2f} units")\n        print(f"Product B (x2): {x2.X:.2f} units")\n    \n    else:\n        print(f"No optimal solution found. Status: {m.status}")\n\nexcept KeyError as e:\n    print(f"Data Error: A required column was not found. {e}", file=sys.stderr)\nexcept pd.errors.EmptyDataError as e:\n    print(f"Data Error: CSV file is empty or unreadable. {e}", file=sys.stderr)\nexcept Exception as e:\n    print(f"An unexpected error occurred: {e}", file=sys.stderr)', 'files': ['Large_Scale_Or_Files/Other_example/model2/production_data.csv']}, {'question': 'A specialty chemical factory produces three premium products (A, B, C) by blending three raw materials (1, 2, 3). The factory supplies custom formulations to high-end manufacturers:\n\nProduct A is a precision industrial lubricant requiring _50% Material 1 and _25% Material 2\nProduct B is an aerospace-grade adhesive requiring _25% Material 1 and _50% Material 2\nProduct C is a general-purpose compound with no material restrictions\nRaw material costs and availability are in raw_materials.csv, while product specifications and selling prices are in products.csv. Determine the optimal blending plan to maximize daily profit.', 'model': '#### Decision Variables\n- $x_{ij}$: kg of material $j$ used in product $i$ \n$i \\in \\{\\text{A}, \\text{B}, \\text{C}\\}, j \\in \\{1,2,3\\}$\n\n#### Objective Function\nMaximize profit: \n$\\quad \\quad \\max \\ 60(x_{A1} + x_{A2} + x_{A3}) + 70(x_{B1} + x_{B2} + x_{B3}) + 60(x_{C1} + x_{C2} + x_{C3})$\n$\\quad \\quad - 65(x_{A1} + x_{B1} + x_{C1}) - 25(x_{A2} + x_{B2} + x_{C2}) - 35(x_{A3} + x_{B3} + x_{C3})$\n\n#### Constraints\n**1. Material Availability** \n$\\quad \\quad x_{A1} + x_{B1} + x_{C1} \\leq 100$ \n$\\quad \\quad x_{A2} + x_{B2} + x_{C2} \\leq 100$ \n$\\quad \\quad x_{A3} + x_{B3} + x_{C3} \\leq 60$ \n\n**2. Product A Specifications** \n$\\quad \\quad x_{A1} \\geq 0.5(x_{A1} + x_{A2} + x_{A3})$ \n$\\quad \\quad x_{A2} \\leq 0.25(x_{A1} + x_{A2} + x_{A3})$ \n\n**3. Product B Specifications** \n$\\quad \\quad x_{B1} \\geq 0.25(x_{B1} + x_{B2} + x_{B3})$ \n$\\quad \\quad x_{B2} \\leq 0.5(x_{B1} + x_{B2} + x_{B3})$ \n\n**4. Non-negativity** \n$\\quad \\quad x_{ij} \\geq 0 \\quad \\forall i,j$', 'code': 'import gurobipy as gp\nimport pandas as pd\nimport numpy as np\nimport sys\nimport re\n\n# Helper function to find columns robustly\n# (Not used in this static model, but included as part of the template)\ndef find_col(df, pattern):\n    for col in df.columns:\n        if re.search(pattern, col, re.IGNORECASE):\n            return col\n    raise KeyError(f"Could not find a column matching pattern \'{{pattern}}\'")\n\ntry:\n    # 1. Load data\n    # (This step is skipped as all parameters are hard-coded\n    # in the mathematical model provided.)\n    data_path = \'{dataset_address}\' # Not used\n\n    # 2. Prepare data structures\n    # (Based on the static values from the problem description)\n    products = [\'A\', \'B\', \'C\']\n    materials = [1, 2, 3] # Using integers for j={1,2,3}\n    \n    # Selling price per kg for each product\n    selling_price = {\'A\': 60, \'B\': 70, \'C\': 60}\n    \n    # Cost per kg for each material\n    material_cost = {1: 65, 2: 25, 3: 35}\n    \n    # Availability in kg for each material\n    availability = {1: 100, 2: 100, 3: 60}\n\n    # 3. Build Gurobi model\n    m = gp.Model(\'MaterialBlending\')\n    \n    # 4. Define variables\n    # x[i,j]: kg of material j used in product i\n    # (Default is continuous and non-negative, which matches x_ij >= 0)\n    x = m.addVars(products, materials, name="x")\n    \n    # 5. Set objective\n    # Maximize (Total Revenue) - (Total Cost)\n    \n    # Helper expressions for total amount of each product\n    total_product = {\n        i: gp.quicksum(x[i, j] for j in materials) for i in products\n    }\n    \n    # Helper expressions for total amount of each material\n    total_material = {\n        j: gp.quicksum(x[i, j] for i in products) for j in materials\n    }\n    \n    # Total Revenue = sum(price[i] * total_product[i])\n    total_revenue = gp.quicksum(\n        selling_price[i] * total_product[i] for i in products\n    )\n    \n    # Total Cost = sum(cost[j] * total_material[j])\n    total_cost = gp.quicksum(\n        material_cost[j] * total_material[j] for j in materials\n    )\n    \n    m.setObjective(total_revenue - total_cost, gp.GRB.MAXIMIZE)\n\n    # 6. Add constraints\n    \n    # 1. Material Availability\n    # x_A1 + x_B1 + x_C1 <= 100  (j=1)\n    # x_A2 + x_B2 + x_C2 <= 100  (j=2)\n    # x_A3 + x_B3 + x_C3 <= 60   (j=3)\n    m.addConstrs(\n        (total_material[j] <= availability[j] for j in materials), \n        name="Material_Availability"\n    )\n    \n    # 2. Product A Specifications\n    # x_A1 >= 0.5 * (total product A)\n    m.addConstr(\n        x[\'A\', 1] >= 0.5 * total_product[\'A\'], \n        name="ProdA_Spec1"\n    )\n    # x_A2 <= 0.25 * (total product A)\n    m.addConstr(\n        x[\'A\', 2] <= 0.25 * total_product[\'A\'], \n        name="ProdA_Spec2"\n    )\n    \n    # 3. Product B Specifications\n    # x_B1 >= 0.25 * (total product B)\n    m.addConstr(\n        x[\'B\', 1] >= 0.25 * total_product[\'B\'], \n        name="ProdB_Spec1"\n    )\n    # x_B2 <= 0.5 * (total product B)\n    m.addConstr(\n        x[\'B\', 2] <= 0.5 * total_product[\'B\'], \n        name="ProdB_Spec2"\n    )\n    \n    # 4. Non-negativity (x_ij >= 0)\n    # This is handled by default in m.addVars\n    \n    # 7. Solve\n    m.optimize()\n    \n    # 8. Print results\n    if m.status == gp.GRB.OPTIMAL:\n        print(f"Optimal total value/cost: {m.objVal:.2f}")\n        \n        print("\\n--- Total Production (kg) ---")\n        for i in products:\n            total_prod_kg = total_product[i].getValue()\n            if total_prod_kg > 1e-6:\n                print(f"Product {i}: {total_prod_kg:.2f} kg")\n\n        print("\\n--- Material Usage (kg) ---")\n        for i in products:\n            for j in materials:\n                if x[i, j].X > 1e-6: # Only print non-zero usage\n                    print(f"  Product {i} uses Material {j}: {x[i, j].X:.2f} kg")\n                    \n        print("\\n--- Total Material Used (kg) ---")\n        for j in materials:\n            total_mat_kg = total_material[j].getValue()\n            if total_mat_kg > 1e-6:\n                print(f"Material {j}: {total_mat_kg:.2f} kg (Available: {availability[j]})")\n                \n    else:\n        print(f"No optimal solution found. Status: {m.status}")\n\nexcept KeyError as e:\n    print(f"Data Error: A required column was not found. {e}", file=sys.stderr)\nexcept pd.errors.EmptyDataError as e:\n    print(f"Data Error: CSV file is empty or unreadable. {e}", file=sys.stderr)\nexcept Exception as e:\n    print(f"An unexpected error occurred: {e}", file=sys.stderr)', 'files': ['Large_Scale_Or_Files/Other_example/model3/products.csv', 'Large_Scale_Or_Files/Other_example/model3/raw_materials.csv']}]}


In [ ]:
CODE_STYLE = {'NRM': 'For example, here is a simple instance for reference:\n\nMathematical Optimization Model:\n\nObjective Function:\n$\\quad \\quad \\max \\quad \\sum_i A_i \\cdot x_i$\nConstraints\n1. Inventory Constraints:\n$\\quad \\quad x_i \\leq I_i, \\quad \x0corall i$\n2. Demand Constraints:\n$x_i \\leq d_i, \\quad \x0corall i$\n3. Startup Constraint:\n$\\sum_i x_i \\geq s$\nRetrieved Information\n$\\small I = [7550, 6244]$\n$\\small A = [149, 389]$\n$\\small d = [15057, 12474]$\n$\\small s = 100$\n\nThe corresponding Python code for this instance is as follows:\n\n```python\nimport gurobipy as gp\nfrom gurobipy import GRB\n\n# Create the model\nm = gp.Model("Product_Optimization")\n\n# Decision variables for the number of units of each product\nx_1 = m.addVar(vtype=GRB.INTEGER, name="x_1") # Number of units of product 1\nx_2 = m.addVar(vtype=GRB.INTEGER, name="x_2") # Number of units of product 2\n\n# Objective function: Maximize 149 x_1 + 389 x_2\nm.setObjective(149 * x_1 + 389 * x_2, GRB.MAXIMIZE)\n\n# Constraints\nm.addConstr(x_1 <= 7550, name="inventory_constraint_1")\nm.addConstr(x_2 <= 6244, name="inventory_constraint_2")\nm.addConstr(x_1 <= 15057, name="demand_constraint_1")\nm.addConstr(x_2 <= 12474, name="demand_constraint_2")\n\n# Non-negativity constraints are implicitly handled by the integer constraints (x_1, x_2 >= 0)\n\n# Solve the model\nm.optimize()', 'RA': "For RA legacy, the complete numerical formulation and original query provide the data contract.\nUse the actual identifiers and coefficients from the formulation; no canonical CSVQA_DATA exists.\nRepresent item counts and discrete units as nonnegative integers unless the original query explicitly\nallows fractional quantities or describes divisible material. Do not infer continuity from scale alone.\nKeep independent resource-by-activity allocations distinct from global activities consuming multiple\nresource dimensions. A separate resource allocation requires that resource in the variable index;\na global activity consuming multiple dimensions requires the correct dimension-specific coefficients.\nEvery capacity constraint must use the corresponding resource decisions or consumption coefficients.\nNever apply the same global expression against every independent warehouse's capacity. Preserve every\nsupplied capacity and per-unit coefficient, explicit business identifier, and query-supported constraint.", 'TP': 'For TP legacy, use the identifiers, coefficients, and dimensions in the numerical formulation.\nPreserve matrix axes and orientation. Never assume a square matrix or invent entity counts.', 'FLP': 'For example, here is a simple instance for reference:\n\nMathematical Optimization Model:\n\nObjective Function:\n$\\quad \\quad \\min \\quad \\sum_{i} \\sum_{j} A_{ij} \\cdot x_{ij} + \\sum_{i} c_i \\cdot y_i$\n\nConstraints\n1. Demand Constraint:\n$\\quad \\quad \\sum_i x_{ij} = d_j, \\quad \\forall j$\n2. Capacity Constraint:\n$\\quad \\quad \\sum_j x_{ij} \\leq M \\cdot y_i, \\quad \\forall i$\n3. Non-negativity:\n$\\quad \\quad x_{ij} \\geq 0, \\quad \\forall i,j$\n4. Binary Requirement:\n$\\quad \\quad y_i \\in \\{0,1\\}, \\quad \\forall i$\n\nRetrieved Information\n$\\small d = [1083, 776, 16214, 553, 17106, 594, 732]$\n$\\small c = [102.33, 94.92, 91.83, 98.71, 95.73, 99.96, 98.16]$\n$\\small A = \\begin{bmatrix}\n1506.22 & 70.90 & 8.44 & 260.27 & 197.47 & 71.71 & 61.19 \\\\  \n1732.65 & 1780.72 & 567.44 & 448.68 & 29.00 & 1484.91 & 963.92 \\\\  \n115.66 & 100.76 & 64.68 & 1324.53 & 64.99 & 134.88 & 2102.83 \\\\  \n1254.78 & 1115.63 & 52.31 & 1036.16 & 892.63 & 1464.04 & 1383.41 \\\\  \n42.90 & 891.01 & 1013.94 & 1128.72 & 58.91 & 42.89 & 1570.31 \\\\  \n0.70 & 139.46 & 70.03 & 79.15 & 1482.00 & 0.91 & 110.46 \\\\  \n1732.30 & 1780.44 & 486.50 & 523.74 & 522.08 & 82.48 & 826.41\n\\end{bmatrix}$\n$\\small M = \\sum_j d_j = 1083 + 776 + 16214 + 553 + 17106 + 594 + 732 = 38058 $\n\nThe corresponding Python code for this instance is as follows:\n\n```python\nimport gurobipy as gp\nfrom gurobipy import GRB\nimport numpy as np\n\n# Data\nd = np.array([1083, 776, 16214, 553, 17106, 594, 732])\nc = np.array([102.33, 94.92, 91.83, 98.71, 95.73, 99.96, 98.16])\nA = np.array([[1506.22, 70.90, 8.44, 260.27, 197.47, 71.71, 61.19],  \n[1732.65, 1780.72, 567.44, 448.68, 29.00, 1484.91, 963.92],  \n[115.66, 100.76, 64.68, 1324.53, 64.99, 134.88, 2102.83],  \n[1254.78, 1115.63, 52.31, 1036.16, 892.63, 1464.04, 1383.41],  \n[42.90, 891.01, 1013.94, 1128.72, 58.91, 42.89, 1570.31],  \n[0.70, 139.46, 70.03, 79.15, 1482.00, 0.91, 110.46],  \n[1732.30, 1780.44, 486.50, 523.74, 522.08, 82.48, 826.41]])\n\n# Create the model\nm = gp.Model("Optimization_Model")\n\n# Decision variables\nx = m.addVars(A.shape[0], A.shape[1], lb=0, name="x")\ny = m.addVars(A.shape[0], vtype=GRB.BINARY, name="y")\n\n# Objective function\nm.setObjective(gp.quicksum(A[i, j]*x[i, j] for i in range(A.shape[0]) for j in range(A.shape[1])) + gp.quicksum(c[i]*y[i] for i in range(A.shape[0])), GRB.MINIMIZE)\n\n# Constraints\nfor j in range(A.shape[1]):\n    m.addConstr(gp.quicksum(x[i, j] for i in range(A.shape[0])) == d[j], name=f"demand_constraint_{j}")\n\nM = 1000000  # large number\nfor i in range(A.shape[0]):\n    m.addConstr(-M*y[i] + gp.quicksum(x[i, j] for j in range(A.shape[1])) <= 0, name=f"M_constraint_{i}")\n\n# Solve the model\nm.optimize()', 'AP': 'For example, here is a simple instance for reference:\n\nMathematical Optimization Model:\n\nObjective Function:\n$\\quad \\quad \\min \\quad \\sum_{i=1}^3 \\sum_{j=1}^3 c_{ij} \\cdot x_{ij}$\n\nConstraints\n1. Row Assignment Constraint:\n$\\quad \\quad \\sum_{j=1}^3 x_{ij} = 1, \\quad \\forall i \\in \\{1,2,3\\}$\n2. Column Assignment Constraint:\n$\\quad \\quad \\sum_{i=1}^3 x_{ij} = 1, \\quad \\forall j \\in \\{1,2,3\\}$\n3. Binary Constraint:\n$\\quad \\quad x_{ij} \\in \\{0,1\\}, \\quad \\forall i,j$\n\nRetrieved Information\n$\\small c = \\begin{bmatrix}\n3000 & 3200 & 3100 \\\\\n2800 & 3300 & 2900 \\\\\n2900 & 3100 & 3000 \n\\end{bmatrix}$\n\nThe corresponding Python code for this instance is as follows:\n\n```python\nimport gurobipy as gp\nfrom gurobipy import GRB\nimport numpy as np\n\n# Data\nc = np.array([\n    [3000, 3200, 3100],\n    [2800, 3300, 2900],\n    [2900, 3100, 3000]\n])\n\n# Create the model\nm = gp.Model("Optimization_Model")\n\n# Decision variables\nx = m.addVars(c.shape[0], c.shape[1], vtype=GRB.BINARY, name="x")\n\n# Objective function\nm.setObjective(gp.quicksum(c[i, j]*x[i, j] for i in range(c.shape[0]) for j in range(c.shape[1])), GRB.MINIMIZE)\n\n# Constraints\nfor i in range(c.shape[0]):\n    m.addConstr(gp.quicksum(x[i, j] for j in range(c.shape[1])) == 1, name=f"row_constraint_{i}")\n\nfor j in range(c.shape[1]):\n    m.addConstr(gp.quicksum(x[i, j] for i in range(c.shape[0])) == 1, name=f"col_constraint_{j}")\n\n# Solve the model\nm.optimize()'}


In [ ]:
METHOD='Abl-FewShotOnly'

# Verbatim questions from RefData.csv, 1-based data rows (19, 62, 56, 92, 87).
CLASS_REFDATA_ROWS = (19, 62, 56, 92, 87)
CLASS_DEMOS = 'Question:\nThere are three best-selling items (P1, P2, P3) on Amazon with the profit w_1,w_2,w_3.There is an independent demand stream for each of the products. The objective of the company is to decide which demands to be fufilled over a __ite sales horizon [0,10] to maximize the total expected revenue from __ed initial inventories. The on-hand inventories for the three items are c_1,c_2,c_3 respectively. During the sales horizon, replenishment is not allowed and there is no any in-transit inventories. Customers who want to purchase P1,P2,P3 arrive at each period accoring to a Poisson process with a_1,a_2,a_3 the arrival rates respectively. Decision variables y_1,y_2,y_3 correspond to the number of requests that the firm plans to fulfill for product 1,2,3. These variables are all positive integers.Profit information: w_1 = 20,w_2 = 10,w_3 = 30; Inventory information: c_1 = 50, c_2 = 70, c_3 =30; Demand information: a_1 = 3, a_2 = 8, a_3 = 2.\nRefData label: Network Revenue Management\nAnswer: NRM\n\nQuestion:\nA 3D printing studio produces prototype parts and precision molds for engineering clients. Each prototype part requires 1.8 hours on the printer and 0.5 hours of technician supervision, bringing in a $25 profit. Each precision mold requires 1.2 hours of printer time and 1.8 hours of technician supervision, bringing in a $45 profit. Each day, the studio has a maximum of 72 hours of printer time and 36 hours of technician supervision available. Determine how many of each item to produce daily to maximize total profit.\nRefData label: Resource Allocation\nAnswer: RA\n\nQuestion:\nIn a city, there are two factories, F1 and F2, capable of producing chocolate, and two supermarkets, S1 and S2, which require one unit of chocolate each. The factories incur a fixed cost, denoted by f_1 and f_2 , respectively, upon operation. Each supermarket needs to import one unit of chocolate. For supermarket S1, the transportation cost per unit of chocolate from factories F1 and F2 is c_11 and c_12, respectively. Similarly, for supermarket S2, the transportation cost per unit from factories F1 and F2 is c_21 and c_22, respectively. The objective is to determine which factories to open so that the demand of all customers is met while minimizing the total cost. The decision variables y_1 and y_2 are binary variables indicating whether factory F1 and F2 are operational (open). The decision variables x_11, x_12 represent the proportion of the total chocolate supply that supermarket S1 imports from factory F1 and F2. The decision variables x_21, x_22 represent the proportion of the total chocolate supply that supermarket S2 imports from factory F1 and F2. Fixed costs for the factories: f_1 = 10, f_2 = 12; Transportation costs per unit of chocolate: c_11 = 4, c_12 = 6; c_21 = 5, c_22 = 7\nRefData label: Facility Location Problem\nAnswer: FLP\n\nQuestion:\nA product can be processed on any one of the four devices A, B, C, or D. The setup completion cost for each device when it is activated, the unit cost of production for the product, and the maximum processing capacity for each device are known as shown in 6.csv. How can the total cost be minimized to produce 2000 units of the product? Try to establish a mathematical model.\nRefData label: Mixture\nAnswer: Mixture\n\nQuestion:\nA factory plans to produce two products (A and B) using three limited resources: steel, iron, and machine hours. The resource consumption per unit of product and profit margins are shown in production_data.csv. Determine the optimal production quantities to maximize total profit while respecting resource constraints.\nRefData label: Others\nAnswer: Others'

MODEL_RULES = ('Use the supplied worked cases for modeling. Any separate code style '
    'reference shows format only and may have different data. '
    'Use only CURRENT CASE files for the current answer. The files are attached '
    'to this API request. No tools are provided.\n\n'+OUTPUT_RULES)

def classify(client,case):
    return plain_call(client,CLASS_RULES+'\n\n'+CLASS_DEMOS,case['query'])

def upload_text_file(client,path,cache):
    import tempfile
    if path in cache:return cache[path]
    # A .txt copy makes the complete CSV available as text. CSV input_file
    # uses a spreadsheet preview capped at 1,000 rows.
    with tempfile.NamedTemporaryFile(mode='w',suffix='.txt',prefix='or_data_',
        encoding='utf-8',delete=False) as tmp:
        temp_path=Path(tmp.name)
        with path.open(encoding='utf-8-sig') as original:
            for line in original:tmp.write(line)
    try:
        with temp_path.open('rb') as data:
            uploaded=client.files.create(file=data,purpose='user_data')
        cache[path]=uploaded.id
        return uploaded.id
    finally:
        temp_path.unlink(missing_ok=True)

def answer_case(client,case,predicted,cache):
    demos=DEMONSTRATIONS[route(predicted)]
    content=[]
    for number,demo in enumerate(demos,1):
        code_section=('Gurobi code:\n'+demo['code']+'\n') if demo['code'] else ''
        content.append({'type':'input_text','text':
            f'Worked case {number}.\nQuestion:\n{demo["question"]}\n'
            f'Mathematical model:\n{demo["model"]}\n'
            +code_section+
            'The following attached files belong only to this worked case.'})
        for raw in demo['files']:
            path=resolve_csv_paths(raw)[0]
            content.append({'type':'input_text','text':'Worked-case CSV: '+rel(path)})
            content.append({'type':'input_file','file_id':upload_text_file(client,path,cache)})
    if route(predicted) in CODE_STYLE:
        content.append({'type':'input_text','text':'Separate code style reference:\n'+CODE_STYLE[route(predicted)]})
    content.append({'type':'input_text','text':
        'CURRENT CASE question:\n'+case['query']+'\n'
        'The following attached CSVs belong only to the CURRENT CASE. '
        'Use the listed repository paths in the generated Gurobi code.'})
    for path in case['files']:
        content.append({'type':'input_text','text':'CURRENT CASE CSV: '+rel(path)})
        content.append({'type':'input_file','file_id':upload_text_file(client,path,cache)})
    response=client.responses.create(model=MODEL,temperature=0,max_output_tokens=32768,
        input=[{'role':'system','content':MODEL_RULES},
               {'role':'user','content':content}])
    return output_text(response),[usage(response)],len(demos)

def run(rows=None,output_dir=None):
    from dotenv import load_dotenv
    from openai import OpenAI
    load_dotenv(ROOT/'.env',override=False)
    if not os.environ.get('OPENAI_API_KEY'):raise RuntimeError('OPENAI_API_KEY is missing')
    selected=set(range(101) if rows is None else rows)
    if not selected.issubset(set(range(101))):raise ValueError('Rows must be 0..100')
    all_cases=cases()
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    folder=Path(output_dir or ROOT/'outputs'/'large_scale_or_ablations'/f'fewshotonly__{stamp}').resolve()
    folder.mkdir(parents=True,exist_ok=True)
    (folder/'answers').mkdir(exist_ok=True)
    log=folder/'records.jsonl'
    done={json.loads(x)['row'] for x in log.read_text().splitlines() if x.strip()} if log.exists() else set()
    client=OpenAI(max_retries=0,timeout=180)
    cache={}
    try:
        for case in all_cases:
            i=case['row']
            if i not in selected or i in done:continue
            rec={'row':i,'gold_type':gold_class(case['type']),'gold_objective':case['gold'],
                 'files':[rel(p) for p in case['files']],'status':'error','tool_calls':0}
            t=time.perf_counter()
            try:
                label,cls_usage=classify(client,case)
                pred=label.strip().strip('`*_ .,:;\n')
                rec['pred_type']=pred
                rec['classification_correct']=pred==rec['gold_type']
                answer,model_usage,n_demos=answer_case(client,case,pred,cache)
                rec['n_demonstrations']=n_demos
                rec['usage']={k:sum(x[k] for x in cls_usage+model_usage) for k in ('input','output')}
                (folder/'answers'/f'row_{i:03d}.txt').write_text(answer,encoding='utf-8')
                source=extract_code(answer)
                objective,solver=solve_code(source)
                rec.update(status='ok',predicted_objective=objective,
                    correct_optimal=math.isclose(objective,case['gold'],rel_tol=1e-4,abs_tol=1e-4),
                    solver_tail=solver)
            except Exception as exc:
                rec.update(correct_optimal=False,error_type=type(exc).__name__,error=str(exc)[-3000:])
            rec['wall_s']=round(time.perf_counter()-t,3)
            with log.open('a',encoding='utf-8') as f:f.write(json.dumps(rec,ensure_ascii=False)+'\n')
            print('row',i,rec['status'],'correct',rec['correct_optimal'])
    finally:
        for file_id in cache.values():
            try:client.files.delete(file_id)
            except Exception as exc:print('Could not clean uploaded file',file_id,exc)
    save_scores(folder,selected)
    return folder


## Preflight

This reads the benchmark and checks every case file without calling an API.

In [ ]:
DATA = check_files()
print("Model:", MODEL, "Method:", METHOD)


## Run

Set `RUN_API` to `True` after choosing rows. Use `ROWS = None` for all 101 cases. Reuse `OUTPUT_DIR` to resume.

In [ ]:
RUN_API = False
ROWS = None
OUTPUT_DIR = None
if RUN_API:
    RUN_DIR = run(rows=ROWS, output_dir=OUTPUT_DIR)
    print(RUN_DIR)
